In [4]:
%pip install -U langchain langchain-community langchain-openai langchain-text-splitters youtube-transcript-api faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [11]:
# Instead of from langchain.text_splitter ...
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Step 1a - Indexing (Document Ingestion)

In [2]:
pip install youtube-transcript-api

  Using cached youtube_transcript_api-1.2.4-py3-none-any.whl.metadata (24 kB)
Using cached youtube_transcript_api-1.2.4-py3-none-any.whl (485 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys

# 1. Force a clean reinstall from the internet, ignoring local cache
!{sys.executable} -m pip uninstall -y youtube-transcript-api
!{sys.executable} -m pip install --no-cache-dir youtube-transcript-api

print("\n--- REINSTALL COMPLETE ---")
print("CRITICAL: Now go to the top menu and click 'Kernel' -> 'Restart'")

Found existing installation: youtube-transcript-api 1.2.4
Uninstalling youtube-transcript-api-1.2.4:
  Successfully uninstalled youtube-transcript-api-1.2.4

--- REINSTALL COMPLETE ---
CRITICAL: Now go to the top menu and click 'Kernel' -> 'Restart'



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "Gfr50f6ZBvo"

try:
    # 1. Create an instance of the API
    api = YouTubeTranscriptApi()
    
    # 2. Use the new .fetch() method instead of .get_transcript()
    # This returns a Transcript object, so we use .to_raw_data() to get the list
    transcript_list = api.fetch(video_id).to_raw_data()
    print(transcript_list[:2])  # Print the first 2 chunks to verify structure
    # Flatten it to plain text for LangChain
    transcript_text = " ".join(chunk["text"] for chunk in transcript_list)
    
    print("SUCCESS! Transcript retrieved.")
    print(f"Sample: {transcript_text[:100]}...")

except Exception as e:
    print(f"Error fetching transcript: {e}")

[{'text': 'the following is a conversation with', 'start': 0.08, 'duration': 3.44}, {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96}]
SUCCESS! Transcript retrieved.
Sample: the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has...


# Step 1b - indexing (Text Splitting)

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Approx 1000 characters per chunk
    chunk_overlap=100,  # 10% overlap so context isn't lost between chunks
    length_function=len,
    is_separator_regex=False,
)

# Split the transcript
chunks = text_splitter.create_documents([transcript_text])

print(f"Split into {len(chunks)} chunks.")
print(f"Example of first chunk:\n{chunks[0].page_content}")

c:\Users\mdala\OneDrive\Desktop\Semester6\GEN_AI\LangChain-\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Split into 149 chunks.
Example of first chunk:
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wro

In [13]:
chunks[100]

Document(metadata={}, page_content="sign a complex topic simply then that's one of the best signs of you understanding it yeah so i can see myself talking trash in the ai system in that way yes uh it gets frustrated how dumb i am and trying to explain something to me i was like well that means you're not intelligent because if you were intelligent you'd be able to explain it simply yeah of course you know there's also the other option of course we could enhance ourselves and and without devices we we are already sort of symbiotic with our compute devices right with our phones and other things and you know this stuff like neural link and etc that could be could could advance that further um so i think there's lots of lots of really amazing possibilities uh that i could foresee from here well let me ask you some wild questions so out there looking for friends do you think there's a lot of alien civilizations out there so i guess this also goes back to your origin of life question too bec

# Step 1c & 1d - Indexing (Embedding Generation and Storing in vector Store)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os


# 1. Initialize Embeddings
embeddings = OpenAIEmbeddings()

# 2. Create Vector Store
vector_store = FAISS.from_documents(chunks, embeddings)

# 3. Save locally (optional)
vector_store.save_local("faiss_index_deepmind")

print("Vector store created successfully!")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}